# Data Ingestion Pipeline — JSON to CSV

This notebook demonstrates how to:
1. Fetch a synthetic JSON dataset from a raw GitHub URL
2. Fetch product and cart data from the DummyJSON API
3. Convert each JSON response into a pandas DataFrame
4. Clean and normalise the data
5. Save the results as CSV files

---

## 1. Imports

In [ ]:
import json
import requests
import pandas as pd
import numpy as np
from pathlib import Path

OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)
print('Libraries loaded.')

## 2. Part A — Synthetic Dataset from GitHub

We read a JSON file hosted on GitHub containing 250 synthetic customer records.

The dataset is stored at: [github.com/atombmn/LuxCoh7_Assignments](https://github.com/atombmn/LuxCoh7_Assignments)

> The script falls back to the local file if the remote fetch fails.

In [ ]:
# Raw GitHub URL for the synthetic dataset
GITHUB_RAW_URL = (
    "https://raw.githubusercontent.com/"
    "atombmn/LuxCoh7_Assignments/main/synthetic_customers.json"
)

try:
    resp = requests.get(GITHUB_RAW_URL, timeout=30)
    resp.raise_for_status()
    raw_data = resp.json()
    print(f"Fetched {len(raw_data)} records from GitHub.")
except Exception as e:
    print(f"GitHub fetch failed ({e}). Loading local file...")
    with open("synthetic_customers.json", encoding="utf-8") as f:
        raw_data = json.load(f)
    print(f"Loaded {len(raw_data)} records from local file.")

df_synth = pd.DataFrame(raw_data)
print(f"Shape: {df_synth.shape}")
df_synth.head()

### 2.1 Data Types Before Cleaning

In [ ]:
df_synth.dtypes

### 2.2 Cleaning

**Strategy:**
- Convert date strings to proper datetime objects (ISO format on export).
- Coerce numeric columns so non-numeric values become NaN.
- Fill missing numeric values with the column median (robust to outliers).
- Fill missing categorical values with `"Unknown"`.
- Fill missing booleans with `False`.

In [ ]:
# Date columns
for col in ["signup_date", "last_login_date"]:
    df_synth[col] = pd.to_datetime(df_synth[col], errors="coerce")

# Numeric columns
numeric_cols = [
    "age", "purchase_count", "total_purchase_amount",
    "average_order_value", "satisfaction_score", "loyalty_points",
]
for col in numeric_cols:
    df_synth[col] = pd.to_numeric(df_synth[col], errors="coerce")
    if df_synth[col].isna().any():
        med = df_synth[col].median()
        count = df_synth[col].isna().sum()
        df_synth[col] = df_synth[col].fillna(med)
        print(f"  Filled {count} NaN in '{col}' with median ({med})")

# Categorical
for col in ["gender", "notes"]:
    df_synth[col] = df_synth[col].fillna("Unknown")

# Boolean
for col in ["is_active", "newsletter_subscribed"]:
    df_synth[col] = df_synth[col].fillna(False)

# Strip whitespace from strings
for col in df_synth.select_dtypes(include="object").columns:
    df_synth[col] = df_synth[col].str.strip()

print("Cleaning complete.")
df_synth.info()

### 2.3 Save to CSV

In [ ]:
synth_path = OUTPUT_DIR / "synthetic_data.csv"
df_synth.to_csv(synth_path, index=False, date_format="%Y-%m-%d")
print(f"Saved {synth_path} ({len(df_synth)} rows)")
df_synth.head(10)

## 3. Part C — DummyJSON Products

Fetch the product catalogue from `https://dummyjson.com/products` and flatten nested fields.

In [ ]:
resp = requests.get("https://dummyjson.com/products?limit=100", timeout=30)
resp.raise_for_status()
products_raw = resp.json()

products_list = products_raw.get("products", products_raw)
df_products = pd.json_normalize(products_list, sep="_")

# Convert list columns to JSON strings for CSV compatibility
list_cols = [c for c in df_products.columns if df_products[c].apply(type).eq(list).any()]
for col in list_cols:
    df_products[col] = df_products[col].apply(
        lambda x: json.dumps(x) if isinstance(x, list) else x
    )

print(f"Products shape: {df_products.shape}")
df_products.head()

In [ ]:
products_path = OUTPUT_DIR / "products.csv"
df_products.to_csv(products_path, index=False)
print(f"Saved {products_path} ({len(df_products)} rows)")

## 4. Part C — DummyJSON Carts

Each cart from `https://dummyjson.com/carts` contains a nested list of products. We explode this into a flat DataFrame where each row is one cart-product pair.

In [ ]:
resp = requests.get("https://dummyjson.com/carts", timeout=30)
resp.raise_for_status()
carts_raw = resp.json()
carts_list = carts_raw.get("carts", carts_raw)

rows = []
for cart in carts_list:
    for product in cart.get("products", []):
        rows.append({
            "cart_id": cart["id"],
            "user_id": cart.get("userId"),
            "cart_total": cart.get("total"),
            "cart_discounted_total": cart.get("discountedTotal"),
            "cart_total_products": cart.get("totalProducts"),
            "cart_total_quantity": cart.get("totalQuantity"),
            "product_id": product.get("id"),
            "product_title": product.get("title"),
            "price": product.get("price"),
            "quantity": product.get("quantity"),
            "product_total": product.get("total"),
            "discount_percentage": product.get("discountPercentage"),
            "discounted_total": product.get("discountedTotal"),
            "thumbnail": product.get("thumbnail"),
        })

df_carts = pd.DataFrame(rows)
print(f"Carts shape: {df_carts.shape}")
df_carts.head()

In [ ]:
carts_path = OUTPUT_DIR / "carts.csv"
df_carts.to_csv(carts_path, index=False)
print(f"Saved {carts_path} ({len(df_carts)} rows)")

## 5. Summary

We have completed the full pipeline:

| Output File | Source | Rows |
|-------------|--------|------|
| `output/synthetic_data.csv` | GitHub raw JSON | 250 |
| `output/products.csv` | DummyJSON `/products` | 30 |
| `output/carts.csv` | DummyJSON `/carts` (exploded) | varies |

All dates are in ISO format (YYYY-MM-DD). Missing values have been imputed as described in the cleaning section.